In [ ]:
!pip install -q openai datasets pandas tqdm scikit-learn huggingface_hub

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import os

HF_TOKEN = userdata.get("HF_TOKEN")
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

login(token=HF_TOKEN)

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
from pathlib import Path
from google.colab import drive

# Mount Google Drive so files survive Colab runtime stop/crash
drive.mount("/content/drive")

# Safe folder in Google Drive
BATCH_DIR = Path("/content/drive/MyDrive/maori_data_finder_openai_batch")
BATCH_DIR.mkdir(parents=True, exist_ok=True)

# Important saved files
BATCH_ID_PATH = BATCH_DIR / "BATCH_ID.txt"
SELECTED_RECORDS_PATH = BATCH_DIR / "selected_records_for_openai_batch.csv"
BATCH_INPUT_PATH = BATCH_DIR / "gpt_5_4_mini_maori_classification_batch.jsonl"
BATCH_OUTPUT_PATH = BATCH_DIR / "gpt_5_4_mini_maori_classification_batch_output.jsonl"
PARSED_RESULTS_PATH = BATCH_DIR / "gpt_5_4_mini_openai_parsed_results.csv"
FINAL_COMPARISON_PATH = BATCH_DIR / "gpt_5_4_mini_openai_label_comparison.csv"

print("Batch folder:", BATCH_DIR)

In [ ]:
from datasets import load_dataset, concatenate_datasets
import pandas as pd

DATASET_REPO_ID = "dinushiTJ/nz_research_commons_classification"

ds_dict = load_dataset(DATASET_REPO_ID)

if "train" in ds_dict and "test" in ds_dict:
    ds_all = concatenate_datasets([ds_dict["train"], ds_dict["test"]])
else:
    split_name = list(ds_dict.keys())[0]
    ds_all = ds_dict[split_name]

df = ds_all.to_pandas()

print(ds_all)
print(ds_all.features)
print("Dataset shape:", df.shape)

display(df.head())

# Convert numeric labels to text labels

In [ ]:
label_col = "classification_label"
label_feature = ds_all.features[label_col]

if hasattr(label_feature, "names"):
    df["classification_label_str"] = df[label_col].apply(
        lambda x: label_feature.int2str(int(x))
    )
else:
    df["classification_label_str"] = df[label_col].astype(str)

display(
    df[[label_col, "classification_label_str"]]
    .drop_duplicates()
    .sort_values(label_col)
)

print(df["classification_label_str"].value_counts())

# Use full dataset or test subset

After the 10-record test works, change this line:

`RUN_FULL_DATASET = True`

In [ ]:
RUN_FULL_DATASET = False   # First test with 10 records then RUN_FULL_DATASET = True
TEST_SIZE = 10

# Change this to True only when you intentionally want to start a new batch
FORCE_NEW_BATCH = False

if SELECTED_RECORDS_PATH.exists() and not FORCE_NEW_BATCH:
    print("Loading previously selected records from Drive...")
    run_df = pd.read_csv(SELECTED_RECORDS_PATH)
else:
    print("Creating new selected records...")

    if RUN_FULL_DATASET:
        run_df = df.copy().reset_index(drop=True)
    else:
        run_df = df.sample(n=TEST_SIZE, random_state=42).reset_index(drop=True)

    run_df["openai_sample_id"] = run_df.index
    run_df.to_csv(SELECTED_RECORDS_PATH, index=False)

print("Rows selected:", len(run_df))
print("Saved selected records to:", SELECTED_RECORDS_PATH)

display(
    run_df[
        [
            "openai_sample_id",
            "record_id_hash",
            "classification_label_str",
            "title",
            "subjects",
            "abstract",
        ]
    ].head()
)

In [ ]:
FORCE_NEW_BATCH = True

if BATCH_ID_PATH.exists():
    BATCH_ID_PATH.unlink()
    print("Deleted old BATCH_ID.txt")

if SELECTED_RECORDS_PATH.exists():
    SELECTED_RECORDS_PATH.unlink()
    print("Deleted old selected records file")

# Build the Māori classification prompt

In [ ]:
import pandas as pd

def clean_text(x):
    if x is None:
        return ""
    if pd.isna(x):
        return ""
    return str(x)


def build_prompt(row):
    title = clean_text(row.get("title", ""))
    authors = clean_text(row.get("authors", ""))
    year = clean_text(row.get("year", ""))
    subjects = clean_text(row.get("subjects", ""))
    abstract = clean_text(row.get("abstract", ""))

    return f"""
You are a specialist academic metadata classifier for New Zealand institutional repository records.

TASK:
Classify the research record below into EXACTLY ONE category:

1. maori_origin
2. non_maori_origin

Definition of maori_origin:
A record is maori_origin when it has a substantive research connection to Māori people, communities, culture, identity, language, history, politics, law, knowledge systems, land, environment, resources, archaeology, material culture, Māori data, Māori funding context, iwi, hapū, whānau, or Te Tiriti-related research context.

A record may qualify as maori_origin if it substantively involves one or more of:

1. PEOPLE AND COMMUNITY
- Māori individuals, whānau, hapū, iwi, or Māori communities as a primary subject, participant group, or beneficiary.
- Māori health, education, economic, justice, social, demographic, or wellbeing outcomes.

2. CULTURE, IDENTITY, AND LANGUAGE
- Māori culture, identity, language, tikanga, mātauranga, whakapapa, waiata, kapa haka, taonga, or cultural expression.
- Māori language presence alone is NOT enough unless the research context is substantively Māori.

3. HISTORY, POLITICS, LAW, AND TE TIRITI
- Māori history, colonisation, Treaty of Waitangi / Te Tiriti, land claims, settlements, governance, law, rights, policy, sovereignty, or Māori political representation.

4. KNOWLEDGE SYSTEMS AND RESEARCH METHODS
- Mātauranga Māori, kaupapa Māori research, Māori methodologies, Māori ethics, Māori data sovereignty, Indigenous data sovereignty with explicit Māori relevance.

5. LAND, ENVIRONMENT, AND RESOURCES
- Māori relationships with whenua, awa, moana, maunga, taiao, conservation, environmental management, climate, freshwater, fisheries, forestry, or resource governance.
- Environmental research is maori_origin only when the Māori connection is substantive, not incidental.

6. ARCHAEOLOGY AND MATERIAL CULTURE
- Māori archaeology, pā sites, artefacts, taonga, settlement, ancestral remains, or pre-European Māori history.

7. DATA, FUNDING, OR INSTITUTIONAL CONTEXT
- Māori datasets, Māori research data, Māori-specific funding, Māori research centres, or institutional Māori research strategy.

Definition of non_maori_origin:
A record is non_maori_origin when it does not have a meaningful Māori research connection.
This includes:
- General New Zealand research with no substantive Māori focus.
- Records that only mention New Zealand locations.
- Records that only include Māori words, place names, or names without Māori research relevance.
- General Indigenous research with no clear Māori connection.
- Environmental, health, education, or social research where Māori are not a substantive focus.

Important decision rule:
Do NOT classify as maori_origin just because the title, abstract, author names, subject terms, or locations contain Māori words. Classify based on the actual research context.

Return ONLY valid JSON in this exact format:
{{
  "label": "maori_origin or non_maori_origin",
  "confidence": 0.0,
  "reason": "brief reason for the classification"
}}

Research record:
Title: {title}
Authors: {authors}
Year: {year}
Subjects: {subjects}
Abstract: {abstract}
""".strip()

# Create Batch JSONL file for gpt-5.4-mini

Reasoning effort can be set in the Responses API with `reasoning={"effort": "low"}` for reasoning models.

In [ ]:
import json

MODEL_NAME = "gpt-5.4-mini"

SYSTEM_INSTRUCTION = (
    "You are a careful academic metadata classifier. "
    "Return only valid JSON with fields: label, confidence, reason. "
    "The label must be either maori_origin or non_maori_origin."
)

with open(BATCH_INPUT_PATH, "w", encoding="utf-8") as f:
    for _, row in run_df.iterrows():
        prompt = build_prompt(row)

        record_id = str(row.get("record_id_hash", row["openai_sample_id"]))
        custom_id = f"record_{row['openai_sample_id']}__{record_id}"

        request = {
            "custom_id": custom_id,
            "method": "POST",
            "url": "/v1/responses",
            "body": {
                "model": MODEL_NAME,
                "reasoning": {
                    "effort": "low"
                },
                "temperature": 0,
                "max_output_tokens": 250,
                "input": [
                    {
                        "role": "system",
                        "content": SYSTEM_INSTRUCTION
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                "text": {
                    "format": {
                        "type": "json_object"
                    }
                }
            }
        }

        f.write(json.dumps(request, ensure_ascii=False) + "\n")

print("Batch JSONL created successfully.")
print("Path:", BATCH_INPUT_PATH)
print("Number of records:", len(run_df))

# Preview first line
with open(BATCH_INPUT_PATH, "r", encoding="utf-8") as f:
    first_line = json.loads(f.readline())

print("\nFirst request preview:")
print(json.dumps(first_line, indent=2, ensure_ascii=False)[:2000])

# Preview first JSONL request

In [ ]:
with open(batch_input_path, "r", encoding="utf-8") as f:
    first_line = json.loads(f.readline())

print(json.dumps(first_line, indent=2, ensure_ascii=False)[:4000])

# Submit the Batch API job

In [ ]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# If BATCH_ID already exists, do not create a duplicate batch
if BATCH_ID_PATH.exists() and not FORCE_NEW_BATCH:
    BATCH_ID = BATCH_ID_PATH.read_text(encoding="utf-8").strip()
    print("Existing batch found. Not submitting a new batch.")
    print("Loaded BATCH_ID:", BATCH_ID)

else:
    print("Uploading batch input JSONL file...")

    batch_input_file = client.files.create(
        file=open(BATCH_INPUT_PATH, "rb"),
        purpose="batch"
    )

    print("Uploaded file ID:", batch_input_file.id)

    print("Creating OpenAI batch job...")

    batch_job = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={
            "project": "maori_data_finder",
            "model": MODEL_NAME,
            "task": "openai_gpt_5_4_mini_cross_provider_consistency_check"
        }
    )

    BATCH_ID = batch_job.id

    # Save immediately to Google Drive
    BATCH_ID_PATH.write_text(BATCH_ID, encoding="utf-8")

    print("New batch submitted successfully.")
    print("Batch ID:", BATCH_ID)
    print("Saved BATCH_ID to:", BATCH_ID_PATH)
    print("Batch status:", batch_job.status)

In [ ]:
BATCH_ID = batch_job.id

batch_id_path = BATCH_DIR / "BATCH_ID.txt"
with open(batch_id_path, "w", encoding="utf-8") as f:
    f.write(BATCH_ID)

print("Saved BATCH_ID to:", batch_id_path)

Save this batch ID somewhere:

`BATCH_ID = batch_job.id`

`print(BATCH_ID)`




# Check batch status

Run this cell later until the status becomes `completed`.

Possible statuses include:

* validating
* in_progress
* finalizing
* completed
* failed
* expired
* cancelled



In [ ]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

if not BATCH_ID_PATH.exists():
    raise FileNotFoundError("BATCH_ID.txt not found. You need to submit the batch first.")

BATCH_ID = BATCH_ID_PATH.read_text(encoding="utf-8").strip()

batch_status = client.batches.retrieve(BATCH_ID)

print("Batch ID:", batch_status.id)
print("Status:", batch_status.status)
print("Request counts:", batch_status.request_counts)
print("Output file ID:", batch_status.output_file_id)
print("Error file ID:", batch_status.error_file_id)

# Download completed batch results

Only run this after status is completed

In [ ]:
batch_status = client.batches.retrieve(BATCH_ID)

print("Current status:", batch_status.status)

if batch_status.status != "completed":
    raise ValueError(f"Batch is not completed yet. Current status: {batch_status.status}")

if BATCH_OUTPUT_PATH.exists():
    print("Output file already downloaded.")
    print("Path:", BATCH_OUTPUT_PATH)

else:
    output_file_id = batch_status.output_file_id

    if output_file_id is None:
        raise ValueError("Batch completed but output_file_id is missing.")

    print("Downloading output file:", output_file_id)

    output_content = client.files.content(output_file_id)

    with open(BATCH_OUTPUT_PATH, "wb") as f:
        f.write(output_content.read())

    print("Downloaded output successfully.")
    print("Saved to:", BATCH_OUTPUT_PATH)

# Download error file if available

In [ ]:
batch_status = client.batches.retrieve(BATCH_ID)

if batch_status.error_file_id:
    error_content = client.files.content(batch_status.error_file_id)
    batch_error_path = BATCH_DIR / "gpt_5_4_mini_maori_classification_batch_errors.jsonl"

    with open(batch_error_path, "wb") as f:
        f.write(error_content.read())

    print("Downloaded error file to:")
    print(batch_error_path)
else:
    print("No error file found.")

  0%|          | 0/2000 [00:00<?, ?it/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  88%|########8 | 2.23MB / 2.54MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Comparison complete.
Local checkpoint: /content/drive/MyDrive/maori_data_finder_llama_groq_checkpoints/llama_groq_label_comparison_checkpoint.csv
Hugging Face repo: dinushiTJ/nz-research-commons-llama-groq-label-comparison
Total completed rows: 2000


# Parse OpenAI batch output

In [ ]:
import json
import pandas as pd

def extract_response_text(batch_row):
    response_body = batch_row.get("response", {}).get("body", {})

    if "output_text" in response_body:
        return response_body["output_text"]

    output = response_body.get("output", [])
    text_parts = []

    for item in output:
        if item.get("type") == "message":
            for content in item.get("content", []):
                if content.get("type") in ["output_text", "text"]:
                    text_parts.append(content.get("text", ""))

    return "\n".join(text_parts).strip()


def safe_parse_json(text):
    try:
        return json.loads(text)
    except Exception:
        return None


parsed_rows = []

with open(BATCH_OUTPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        batch_row = json.loads(line)

        custom_id = batch_row.get("custom_id")
        error = batch_row.get("error")

        output_text = extract_response_text(batch_row)
        parsed_json = safe_parse_json(output_text)

        if parsed_json is None:
            openai_label = None
            openai_confidence = None
            openai_reason = None
            openai_error = f"JSON_PARSE_ERROR: {output_text[:500]}"
        else:
            openai_label = parsed_json.get("label")
            openai_confidence = parsed_json.get("confidence")
            openai_reason = parsed_json.get("reason")
            openai_error = None

        if error is not None:
            openai_error = str(error)

        parsed_rows.append(
            {
                "custom_id": custom_id,
                "openai_label": openai_label,
                "openai_confidence": openai_confidence,
                "openai_reason": openai_reason,
                "openai_error": openai_error,
                "openai_raw_output": output_text
            }
        )

openai_results = pd.DataFrame(parsed_rows)

openai_results.to_csv(PARSED_RESULTS_PATH, index=False)

print("Parsed rows:", len(openai_results))
print("Saved parsed results to:", PARSED_RESULTS_PATH)

print("\nLabel counts:")
print(openai_results["openai_label"].value_counts(dropna=False))

print("\nErrors:", openai_results["openai_error"].notna().sum())

display(openai_results.head())

# Recover sample ID and merge with original dataset

In [ ]:
def extract_sample_id(custom_id):
    try:
        left = custom_id.split("__")[0]
        return int(left.replace("record_", ""))
    except Exception:
        return None


# Reload from Drive if needed
if "run_df" not in globals():
    run_df = pd.read_csv(SELECTED_RECORDS_PATH)

if "openai_results" not in globals():
    openai_results = pd.read_csv(PARSED_RESULTS_PATH)

openai_results["openai_sample_id"] = openai_results["custom_id"].apply(extract_sample_id)

comparison_results = run_df.merge(
    openai_results,
    on="openai_sample_id",
    how="left"
)

comparison_results["exact_match"] = (
    comparison_results["classification_label_str"] == comparison_results["openai_label"]
)

comparison_results["comparison_provider"] = "OpenAI"
comparison_results["comparison_model"] = MODEL_NAME
comparison_results["comparison_type"] = "full_dataset_cross_provider_consistency_check"
comparison_results["prompt_version"] = "maori_origin_metadata_prompt_v1"
comparison_results["reasoning_effort"] = "low"
comparison_results["temperature"] = 0

comparison_results.to_csv(FINAL_COMPARISON_PATH, index=False)

print("Merged shape:", comparison_results.shape)
print("Saved final comparison to:", FINAL_COMPARISON_PATH)

print("\nExact match counts:")
print(comparison_results["exact_match"].value_counts(dropna=False))

display(
    comparison_results[
        [
            "openai_sample_id",
            "record_id_hash",
            "classification_label_str",
            "openai_label",
            "openai_confidence",
            "exact_match",
            "openai_reason",
            "title"
        ]
    ].head()
)

# Save comparison results to CSV

In [ ]:
output_csv_path = BATCH_DIR / "gpt_5_4_mini_openai_label_comparison.csv"

comparison_results.to_csv(output_csv_path, index=False)

print("Saved to:")
print(output_csv_path)

Optional Google Drive copy:

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/maori_data_finder_openai_batch")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

drive_csv_path = DRIVE_DIR / "gpt_5_4_mini_openai_label_comparison.csv"
comparison_results.to_csv(drive_csv_path, index=False)

print("Saved to Google Drive:")
print(drive_csv_path)

# Summary table

In [ ]:
total_records = len(comparison_results)
successful_records = comparison_results["openai_label"].notna().sum()
failed_records = comparison_results["openai_label"].isna().sum()

matched_records = comparison_results["exact_match"].sum()
mismatched_records = successful_records - matched_records

summary_table = pd.DataFrame([
    {
        "Metric": "Total records",
        "Count": total_records,
        "Percentage": 100.00,
    },
    {
        "Metric": "Successfully classified by OpenAI",
        "Count": successful_records,
        "Percentage": successful_records / total_records * 100 if total_records > 0 else 0,
    },
    {
        "Metric": "Failed / missing OpenAI label",
        "Count": failed_records,
        "Percentage": failed_records / total_records * 100 if total_records > 0 else 0,
    },
    {
        "Metric": "Exact matches",
        "Count": matched_records,
        "Percentage": matched_records / successful_records * 100 if successful_records > 0 else 0,
    },
    {
        "Metric": "Mismatches",
        "Count": mismatched_records,
        "Percentage": mismatched_records / successful_records * 100 if successful_records > 0 else 0,
    },
])

display(summary_table)

# Confusion matrix / label comparison table

In [ ]:
label_comparison_counts = pd.crosstab(
    comparison_results["classification_label_str"],
    comparison_results["openai_label"],
    rownames=["Reference label"],
    colnames=["OpenAI gpt-5.4-mini label"],
    margins=True,
)

display(label_comparison_counts)

# Classification report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

eval_df = comparison_results[
    comparison_results["openai_label"].isin(["maori_origin", "non_maori_origin"])
].copy()

y_true = eval_df["classification_label_str"]
y_pred = eval_df["openai_label"]

print(classification_report(
    y_true,
    y_pred,
    labels=["maori_origin", "non_maori_origin"],
    digits=4
))

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=["maori_origin", "non_maori_origin"]
)

cm_df = pd.DataFrame(
    cm,
    index=["True maori_origin", "True non_maori_origin"],
    columns=["Pred maori_origin", "Pred non_maori_origin"]
)

display(cm_df)

# View mismatches

In [ ]:
mismatch_table = comparison_results[
    (comparison_results["openai_label"].notna()) &
    (comparison_results["classification_label_str"] != comparison_results["openai_label"])
].copy()

print("Mismatches:", len(mismatch_table))

display(
    mismatch_table[
        [
            "openai_sample_id",
            "record_id_hash",
            "classification_label_str",
            "openai_label",
            "openai_confidence",
            "openai_reason",
            "title",
            "subjects",
            "abstract",
        ]
    ].head(50)
)

# Push full comparison dataset to Hugging Face

In [ ]:
from datasets import Dataset
from huggingface_hub import login
from google.colab import userdata
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

HF_OUTPUT_REPO_ID = "dinushiTJ/nz-research-commons-openai-gpt-5-4-mini-label-comparison"

comparison_results = pd.read_csv(FINAL_COMPARISON_PATH)

clean_df = comparison_results.copy()

for col in clean_df.columns:
    if clean_df[col].dtype == "object":
        clean_df[col] = clean_df[col].fillna("").astype(str)

hf_dataset = Dataset.from_pandas(clean_df, preserve_index=False)

hf_dataset.push_to_hub(HF_OUTPUT_REPO_ID)

print("Uploaded to Hugging Face:")
print(HF_OUTPUT_REPO_ID)

# Push mismatches only to Hugging Face

In [ ]:
HF_MISMATCH_REPO_ID = "dinushiTJ/nz-research-commons-openai-gpt-5-4-mini-label-mismatches"

mismatch_clean = mismatch_table[available_keep_cols].copy()

for col in mismatch_clean.columns:
    if mismatch_clean[col].dtype == "object":
        mismatch_clean[col] = mismatch_clean[col].fillna("").astype(str)

mismatch_hf = Dataset.from_pandas(
    mismatch_clean,
    preserve_index=False
)

mismatch_hf.push_to_hub(HF_MISMATCH_REPO_ID)

print("Pushed mismatches to Hugging Face:")
print(HF_MISMATCH_REPO_ID)